<a href="https://colab.research.google.com/github/gurliv21/deforestation-domain-adaptation/blob/main/deforestation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
from torch.utils.data import Dataset
import numpy as np
import os

class DeforestationDataset(Dataset):
  def __init__(self, tile_ids, image_dir, mask_dir):
        self.tile_ids = tile_ids
        self.image_dir = image_dir
        self.mask_dir = mask_dir

  def __len__(self):
    return len(self.tile_ids)

  def __getitem__(self,idx):
    tile_id = self.tile_ids[idx]

    img = np.load(os.path.join(self.image_dir, f"{tile_id}.npy"))
    mask = np.load(os.path.join(self.mask_dir, f"{tile_id}.npy"))

    img = self._fix_size(img, 256)
    mask = self._fix_size(mask, 256)

    image_tensor = torch.from_numpy(img).permute(2, 0, 1).float()
    mask_tensor = torch.from_numpy(mask).permute(2,0,1).float()

    return image_tensor, mask_tensor

  def _fix_size(self, arr, target_size):
    h, w = arr.shape[0], arr.shape[1]

    # Crop if too big
    arr = arr[:target_size, :target_size]

    # Pad if too small
    pad_h = target_size - arr.shape[0]
    pad_w = target_size - arr.shape[1]
    if pad_h > 0 or pad_w > 0:
        arr = np.pad(arr, ((0, max(pad_h, 0)), (0, max(pad_w, 0)), (0, 0)), mode='constant')

    return arr


In [ ]:
base_dir="/content/drive/MyDrive/deforestation_project_02"
tiles_path = base_dir+"/tiles/masks/tile_0000_0000.npy"

# print(os.listdir(tiles_path)[:10])

data = np.load(tiles_path)
image_tensor = torch.from_numpy(data).permute(2, 0, 1).float()
print(image_tensor.shape)


torch.Size([1, 257, 259])


In [ ]:
SPLIT_DIR =base_dir+"/splits"
IMAGE_DIR =base_dir+"/tiles/images"
MASK_DIR =base_dir+"/tiles/masks"

with open(os.path.join(SPLIT_DIR, 'train.txt')) as f:
    train_ids = f.read().splitlines()

train_dataset = DeforestationDataset(train_ids,IMAGE_DIR,MASK_DIR)

with open(os.path.join(SPLIT_DIR, 'val.txt')) as f:
    val_ids = f.read().splitlines()

val_dataset = DeforestationDataset(val_ids, IMAGE_DIR, MASK_DIR)

with open(os.path.join(SPLIT_DIR, 'test.txt')) as f:
    test_ids = f.read().splitlines()

test_dataset = DeforestationDataset(test_ids, IMAGE_DIR, MASK_DIR)

print("Dataset size" , len(train_dataset))
img, mask = train_dataset[0]
print("Image tensor shape:", img.shape, img.dtype)
print("Mask tensor shape:", mask.shape, mask.dtype)

Dataset size 315
Image tensor shape: torch.Size([3, 256, 256]) torch.float32
Mask tensor shape: torch.Size([1, 256, 256]) torch.float32


In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE=8

train_dataset = DeforestationDataset(train_ids,IMAGE_DIR,MASK_DIR)
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)

images, masks = next(iter(train_loader))
print("Batch images shape:", images.shape)
print("Batch masks shape:", masks.shape)

Batch images shape: torch.Size([8, 3, 256, 256])
Batch masks shape: torch.Size([8, 1, 256, 256])


In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
      self.in_channels = in_channels
      self.out_channels = out_channels
      super().__init__()

      self.conv = nn.Sequential(
          nn.Conv2d(in_channels,out_channels,kernel_size=3,padding=1),
          nn.ReLU(),
          nn.Conv2d(out_channels,out_channels,kernel_size=3,padding=1),
          nn.ReLU(),

      )
    def forward(self,x):
       return self.conv(x)


test_block = DoubleConv(in_channels=3, out_channels=16)
test_input = torch.randn(8, 3, 256, 256)
test_output = test_block(test_input)
print("Output shape:", test_output.shape)

Output shape: torch.Size([8, 16, 256, 256])


In [ ]:
class UNet(nn.Module):
  def __init__(self, in_channels=3, out_channels=1):
    super().__init__()

    #encoding

    self.enc1 = DoubleConv(in_channels, 64)
    self.enc2 = DoubleConv(64, 128)
    self.enc3 = DoubleConv(128, 256)
        #pooling
    self.pool = nn.MaxPool2d(2)

    #bottleneck

    self.bottleneck = DoubleConv(256, 512)

    #decoder

    self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
    self.dec3 = DoubleConv(512, 256)

    self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
    self.dec2 = DoubleConv(256, 128)

    self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
    self.dec1 = DoubleConv(128, 64)

    self.final = nn.Conv2d(64, out_channels, kernel_size=1)

  def forward(self, x):
        # Encoder path
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        # Bottleneck
        b = self.bottleneck(self.pool(e3))

        # Decoder path
        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.final(d1)



model = UNet(in_channels=3, out_channels=1)
test_input = torch.randn(1, 3, 256, 256)
test_output = model(test_input)
print("Output shape:", test_output.shape)




Output shape: torch.Size([1, 1, 256, 256])


In [ ]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(in_channels=3, out_channels=1).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_model(model,loader,optimizer,criterion,device):
  model.train()
  total_loss =0
  for batch_idx, (images,masks) in enumerate(loader):
    images =images.to(device)
    masks = masks.to(device)
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs,masks)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    if batch_idx % 10 == 0:
            print(f"  Batch {batch_idx}/{len(loader)}, loss: {loss.item():.4f}")

  avg_loss =total_loss/len(loader)
  return avg_loss


epoch_loss = train_model(model,train_loader, optimizer,criterion,device)
torch.save(model.state_dict(), base_dir+'/model_checkpoint.pth')
print("Average loss for this epoch:", epoch_loss)

  Batch 0/40, loss: 2.3736
  Batch 10/40, loss: 0.6271
  Batch 20/40, loss: 0.5898
  Batch 30/40, loss: 0.4817
Average loss for this epoch: 1.29111451767385


In [ ]:
def validate_model(model,loader,criterion,device):
  model.eval()
  total_loss =0

  with torch.no_grad():
    for images, masks in loader:
      images = images.to(device)
      masks = masks.to(device)

      outputs = model(images)
      loss = criterion(outputs, masks)
      total_loss += loss.item()
  avg_loss = total_loss/len(loader)
  return avg_loss


val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

val_loss = validate_model(model,val_loader,criterion,device)
print("Validation loss:", val_loss)

Validation loss: 1.1960936337709427


In [ ]:
NUM_EPOCHS =15
CHECKPOINT_PATH = base_dir+'/model_checkpoint.pth'

for epoch in range(NUM_EPOCHS):
  train_loss = train_model(model,train_loader,optimizer,criterion,device)
  val_loss = validate_model(model,val_loader,criterion,device)

  print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Train loss: {train_loss:.4f}, Val loss: {val_loss:.4f}")

  torch.save(model.state_dict(), CHECKPOINT_PATH)

  Batch 0/40, loss: 1.3439
  Batch 10/40, loss: 1.2753
  Batch 20/40, loss: 1.0883
  Batch 30/40, loss: 1.1357
Epoch 1/15 - Train loss: 1.2270, Val loss: 1.1961
  Batch 0/40, loss: 1.1840
  Batch 10/40, loss: 1.0646
  Batch 20/40, loss: 1.1550
  Batch 30/40, loss: 1.2039
Epoch 2/15 - Train loss: 1.2283, Val loss: 1.1961
  Batch 0/40, loss: 1.2835
  Batch 10/40, loss: 1.2447


In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    images, masks = next(iter(val_loader))
    images = images.to(device)
    outputs = model(images)
    predictions = torch.sigmoid(outputs).cpu()


fig, axes = plt.subplots(4, 3, figsize=(15, 20))

for idx in range(4):
    img = images[idx].cpu().permute(1, 2, 0).numpy()
    real_mask = masks[idx].squeeze().numpy()
    pred_mask = predictions[idx].squeeze().numpy()

    axes[idx, 0].imshow(img / img.max())
    axes[idx, 0].set_title(f"Image {idx}")
    axes[idx, 1].imshow(real_mask, cmap='Reds')
    axes[idx, 1].set_title(f"Real mask {idx}")
    axes[idx, 2].imshow(pred_mask, cmap='Reds')
    axes[idx, 2].set_title(f"Predicted mask {idx}")

    print(f"Tile {idx} - Predicted probability range: {pred_mask.min():.4f} to {pred_mask.max():.4f}")

plt.tight_layout()
plt.show()

In [ ]:
def evaluate_model(model, loader, device, threshold=0.5):
    model.eval()

    total_intersection = 0
    total_union = 0
    total_tp = 0
    total_fp = 0
    total_fn = 0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            probs = torch.sigmoid(outputs)
            preds = (probs > threshold).float()  # convert to 0/1 hard predictions

            # Flatten to simplify pixel-wise comparison
            preds_flat = preds.view(-1)
            masks_flat = masks.view(-1)

            intersection = (preds_flat * masks_flat).sum().item()
            union = ((preds_flat + masks_flat) > 0).float().sum().item()

            tp = intersection
            fp = (preds_flat * (1 - masks_flat)).sum().item()
            fn = ((1 - preds_flat) * masks_flat).sum().item()

            total_intersection += intersection
            total_union += union
            total_tp += tp
            total_fp += fp
            total_fn += fn

    iou = total_intersection / total_union if total_union > 0 else 0
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {'iou': iou, 'precision': precision, 'recall': recall, 'f1': f1}

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
results = evaluate_model(model, test_loader, device)

#How much do my predicted areas overlap the real areas?
print(f"Test IoU: {results['iou']:.4f}")
#of everything  my model called deforestation how many time it was right
print(f"Test Precision: {results['precision']:.4f}")
#Of all the deforestation that actually exists in the label, how much did my model find
print(f"Test Recall: {results['recall']:.4f}")
# F1 combines Precision + Recall into one number.
print(f"Test F1: {results['f1']:.4f}")